# Opportunity — multi-source job search

Search multiple public job feeds with an optional keyword and publication timeframe. Leave `QUERY` empty to browse latest available listings, sorted newest first; use `DAYS = None` to skip the date filter. Remote-only filtering is optional; by default, onsite and hybrid listings are kept when a source provides them. This notebook clones the Python project into the Colab runtime. No required third-party packages or API keys are needed.

Example: search for **data entry jobs posted in the last 5 days**. A date filter can only work with publication dates and history exposed by each source.

In [ ]:
import os
import subprocess
import sys
import importlib
import inspect

REPO_DIR = "/content/opportunity"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["rm", "-rf", REPO_DIR], check=False)
    subprocess.run(["git", "clone", "https://github.com/AgentCalmz/opportunity.git", REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
# Drop any older module object retained in the notebook kernel, then import the checked-out file.
sys.modules.pop("opportunity", None)
importlib.invalidate_caches()
import opportunity

assert os.path.realpath(opportunity.__file__) == os.path.realpath(os.path.join(REPO_DIR, "opportunity.py")), opportunity.__file__
for source_id, (_label, runner) in opportunity.SOURCE_RUNNERS.items():
    inspect.signature(runner).bind("data entry", 5)
search_jobs = opportunity.search_jobs
print_report = opportunity.print_report
print("Loaded current code from:", opportunity.__file__)
print("Adapter signature checks passed:", ", ".join(opportunity.SOURCE_RUNNERS))

In [ ]:
# Change these search settings as needed. Use days=None to skip the date filter.
QUERY = ""  # blank = latest listings, no keyword filter; or e.g. "data entry"
DAYS = None              # None = all dates currently available; or e.g. 1, 5, 7, 30
LOCATION = None           # None = all available locations; or e.g. "Nigeria"
MAX_RESULTS = 5000  # return all matching jobs the source budgets can retrieve
REMOTE_ONLY = False  # False includes onsite/hybrid jobs when available; True filters to remote

report = search_jobs(
    QUERY,
    days=DAYS,
    location=LOCATION,
    remote_only=REMOTE_ONLY,
    max_results=MAX_RESULTS,
)
print_report(report)

In [ ]:
# Optional: inspect matched results as a table. Pandas is normally preinstalled in Colab.
import pandas as pd

columns = [
    "title", "company", "location", "published_at",
    "employment_type", "seniority", "salary_min", "salary_max",
    "salary_currency", "job_url", "apply_url", "source_name"
]
df = pd.DataFrame(report["results"])
display(df[[column for column in columns if column in df.columns]])

In [ ]:
# Optional: save the complete report, including full job descriptions, to JSON.
import json

OUTPUT_FILE = "opportunity_results.json"
with open(OUTPUT_FILE, "w", encoding="utf-8") as file:
    json.dump(report, file, ensure_ascii=False, indent=2)
print("Saved:", os.path.abspath(OUTPUT_FILE))

# Uncomment these lines to download the JSON to your computer from Colab:
# from google.colab import files
# files.download(OUTPUT_FILE)